# AgentCore Memory for the UnderCover Agent

AgentCore Memory is a managed store with two sides. The short-term side keeps the events of a conversation, by actor and session. The long-term side keeps records that can be searched by meaning, organized into namespaces. Records get there in two ways: a strategy reads the events and writes what is worth keeping, or we write records ourselves.

This notebook uses both. It loads UnderCover's policy wording into a `/wording` namespace and searches it, which is the job Qdrant does in Week 2. Then it records a handler's conversation and watches the strategy turn it into notes in that handler's own namespace.

What it creates: one memory, `undercover_memory`, tagged `project=undercover`. Memory bills for what is stored and searched, with no hourly charge: events at 25 cents per thousand, records at 75 cents per thousand a month, searches at 50 cents per thousand. This notebook costs a fraction of a cent.

## 1. Setup

Two clients: `bedrock-agentcore-control` creates and deletes the memory, and `bedrock-agentcore` reads and writes what is in it.

In [ ]:
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import boto3

MEMORY_NAME = "undercover_memory"
STRATEGY_NAME = "handler_notes"
WORDING_NAMESPACE = "/wording"
PROJECT_TAG = {"project": "undercover"}

session = boto3.Session()
agentcore = session.client("bedrock-agentcore-control")
memory_client = session.client("bedrock-agentcore")
tagging = session.client("resourcegroupstaggingapi")

identity = session.client("sts").get_caller_identity()
print(f"Working as {identity['Arn']} in {session.region_name}")

## 2. Create the memory

The memory has one strategy, a semantic one, which reads each conversation and keeps the facts worth remembering. Its namespace template, `/handlers/{actorId}`, gives every handler a namespace of their own. Events expire after seven days, the shortest the service allows. Creating a memory takes two to three minutes.

In [ ]:
def find_memories():
    pages = agentcore.get_paginator("list_memories").paginate()
    return [m["id"] for page in pages for m in page["memories"] if m["id"].startswith(MEMORY_NAME)]


existing = find_memories()

if existing:
    memory_id = existing[0]
else:
    memory_id = agentcore.create_memory(
        name=MEMORY_NAME,
        description="What the UnderCover Agent knows and remembers",
        eventExpiryDuration=7,
        memoryStrategies=[
            {"semanticMemoryStrategy": {"name": STRATEGY_NAME, "namespaces": ["/handlers/{actorId}"]}}
        ],
        tags=PROJECT_TAG,
    )["memory"]["id"]

start = time.time()
while (memory := agentcore.get_memory(memoryId=memory_id)["memory"])["status"] == "CREATING":
    time.sleep(5)

print(f"{memory_id}: {memory['status']} after {time.time() - start:.0f}s")
for strategy in memory["strategies"]:
    print(f"  strategy {strategy['name']}: {strategy['status']}, namespaces {strategy['namespaces']}")

## 3. Load the policy wording

The wording is what the business gives the agent, so it is written straight into the long-term side with `batch_create_memory_records`, one record for each numbered section, all in the `/wording` namespace. No strategy is involved. AgentCore embeds each record, and that takes about a minute, so the cell waits until a search finds something.

In [ ]:
wording = Path("../data/policy_wording.md").read_text()
sections = [part.strip() for part in re.split(r"\n(?=## )", wording) if part.strip().startswith("## ")]

print(f"{len(sections)} sections, the longest {max(len(s) for s in sections):,} characters (a record holds 16,000)")


def search(namespace, query, top_k=3):
    return memory_client.retrieve_memory_records(
        memoryId=memory_id, namespace=namespace, searchCriteria={"searchQuery": query, "topK": top_k}
    )["memoryRecordSummaries"]


already = memory_client.list_memory_records(memoryId=memory_id, namespace=WORDING_NAMESPACE)["memoryRecordSummaries"]

if already:
    print("The wording is already loaded")
else:
    records = [
        {
            "requestIdentifier": f"wording-{number}",
            "namespaces": [WORDING_NAMESPACE],
            "content": {"text": section},
            "timestamp": datetime.now(timezone.utc),
        }
        for number, section in enumerate(sections, 1)
    ]
    result = memory_client.batch_create_memory_records(memoryId=memory_id, records=records)
    print(f"Written {len(result['successfulRecords'])}, failed {len(result['failedRecords'])}")

    start = time.time()
    while not search(WORDING_NAMESPACE, "baggage"):
        time.sleep(5)
    print(f"Searchable {time.time() - start:.0f}s after the write")

## 4. Search it

A search is a question in plain words. Each result comes back with a score, and the first line of each section is its heading.

In [ ]:
questions = [
    "How much is covered for a delayed bag?",
    "My passport was stolen abroad",
    "Is a hospital visit covered and what is the deductible?",
    "laptop stolen from a rental car",
]

for question in questions:
    start = time.perf_counter()
    results = search(WORDING_NAMESPACE, question)
    print(f"{question}  ({time.perf_counter() - start:.2f}s)")
    for result in results:
        print(f"    {result['score']:.3f}  {result['content']['text'].splitlines()[0]}")

## 5. What a handler tells the agent

A conversation is stored as events, under the handler who had it and a session id. Another handler cannot see those events.

In [ ]:
handler = "handler-dana"
session_id = f"session-{uuid.uuid4().hex[:8]}"

conversation = [
    ("USER", "When a claim is for a delayed bag, I always want the airline's delay report checked before anything is paid."),
    ("ASSISTANT", "Understood. For baggage delay claims I will check for the airline's delay report first."),
    ("USER", "Also, I prefer payout amounts rounded to the nearest dollar in summaries."),
]

for role, text in conversation:
    memory_client.create_event(
        memoryId=memory_id,
        actorId=handler,
        sessionId=session_id,
        eventTimestamp=datetime.now(timezone.utc),
        payload=[{"conversational": {"content": {"text": text}, "role": role}}],
    )

for actor in (handler, "handler-sam"):
    events = memory_client.list_events(memoryId=memory_id, actorId=actor, sessionId=session_id)["events"]
    print(f"{actor}: {len(events)} events in {session_id}")

The strategy works in the background. A minute or two after the conversation, its notes appear in the handler's namespace, written as facts about what this handler wants.

In [ ]:
handler_namespace = f"/handlers/{handler}"

start = time.time()
while not (notes := memory_client.list_memory_records(memoryId=memory_id, namespace=handler_namespace)["memoryRecordSummaries"]):
    time.sleep(5)

print(f"Notes appeared after {time.time() - start:.0f}s")
for note in notes:
    print(f"    {note['content']['text']}")

The agent can now search that namespace before it works on a claim for this handler. The same search in another handler's namespace finds nothing.

In [ ]:
for namespace in (handler_namespace, "/handlers/handler-sam"):
    results = search(namespace, "What should I check on a delayed baggage claim?")
    print(namespace)
    for result in results:
        print(f"    {result['score']:.3f}  {result['content']['text']}")
    if not results:
        print("    nothing")

## 6. What now exists in the account

The memory is found by name, then by the `project=undercover` tag.

In [ ]:
def find_tagged():
    tag_filter = [{"Key": key, "Values": [value]} for key, value in PROJECT_TAG.items()]
    pages = tagging.get_paginator("get_resources").paginate(TagFilters=tag_filter)
    return [r["ResourceARN"] for page in pages for r in page["ResourceTagMappingList"]]


def show_resources():
    print(f"{'Memories:':<28}{', '.join(find_memories()) or 'none'}")
    print(f"{'Tagged project=undercover:':<28}{', '.join(find_tagged()) or 'none'}")


show_resources()

## 7. Teardown (optional)

Run this when you are done. It deletes the memories whose names start with `undercover_memory`, with everything in them, and leaves the other labs alone.

In [ ]:
for doomed_id in find_memories():
    agentcore.delete_memory(memoryId=doomed_id)
    print(f"Deleting memory {doomed_id}")

while find_memories():
    time.sleep(5)

show_resources()